In [ ]:
import json
import os
import sys
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from scraper import scrap_website, scrap_website_links

In [37]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"

In [48]:
links = scrap_website_links("https://www.udemy.com")
links

['/',
 '/cart/',
 '/ai-roleplay/',
 '/mobile/',
 '/invite/',
 '/support/',
 '/udemy-business/?ref=ub_header_home&locale=en_US',
 '/udemy-business/plans/?ref=ufb_header_plans&locale=en_US',
 '/udemy-business/request-demo-mx/?ref=ufb_header_demo&locale=en_US',
 '#main-content-anchor',
 '/',
 '/browse/certification/',
 '/browse/certification/comptia-certifications/',
 '/browse/certification/aws-certifications/',
 '/browse/certification/project-management-institute-pmi-certifications/',
 '/featured-topics/',
 '/udemy-business/?locale=en_US&path=request-demo-mx%2F&ref=footer-ad',
 '/']

In [39]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [40]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = scrap_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [47]:
print(get_links_user_prompt("https://www.udemy.com"))


Here is the list of links on the website https://www.udemy.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/cart/
/ai-roleplay/
/mobile/
/invite/
/support/
/udemy-business/?ref=ub_header_home&locale=en_US
/udemy-business/plans/?ref=ufb_header_plans&locale=en_US
/udemy-business/request-demo-mx/?ref=ufb_header_demo&locale=en_US
#main-content-anchor
/
/browse/certification/
/browse/certification/comptia-certifications/
/browse/certification/aws-certifications/
/browse/certification/project-management-institute-pmi-certifications/
/featured-topics/
/udemy-business/?locale=en_US&path=request-demo-mx%2F&ref=footer-ad
/


In [49]:
def select_relevant_links(url):
    openai = OpenAI(base_url = OLLAMA_BASE_URL, api_key="ollama")
    response = openai.chat.completions.create(
        model = MODEL,
        messages = [
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [50]:
select_relevant_links("https://www.udemy.com")

{'links': [{'type': 'about page', 'url': 'https://www.udemy.com/about'},
  {'type': 'company page', 'url': 'https://www.udemy.com/about'},
  {'type': 'features page', 'url': 'https://www.udemy.com/featured-topics'},
  {'type': 'careers/jobs page', 'url': 'https://www.udemy.com/careers'}]}

### TO MAKE BROCHURE

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = scrap_website(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += scrap_website(link["url"])
    return result